# Reading CSVW-described data with `csvw`

Loads a CSV through its CSVW metadata, checks it against the schema, prints a few rows and turns it into a pandas DataFrame.

In [1]:
from pathlib import Path

import pandas as pd
from csvw import CSVW

METADATA = Path("../csvw/waterinfo_289423042_csvw.json")

## 1. Load and validate

`validate=True` reads the CSV and checks every row against the schema (datatypes, required values, header), the same check `csvwvalidate` runs.

In [2]:
data = CSVW(str(METADATA), validate=True)
print("valid:", data.is_valid)
for w in data.warnings:
    print("warning:", w.message)

table = data.tables[0]
print("CSV:", table.url)
for col in table.tableSchema.columns:
    print(f"  {col.name:26} {col.datatype.base:9} header: {col.titles.getfirst()}")

valid: True
CSV: ../data/waterinfo_289423042.csv
  ts_id                      integer   header: ts_id
  timestamp                  dateTime  header: Timestamp
  value                      double    header: Value
  interpolation_type         integer   header: Interpolation Type
  quality_code               integer   header: Quality Code
  quality_code_name          string    header: Quality Code Name
  quality_code_description   string    header: Quality Code Description
  quality_code_color         string    header: Quality Code Color


## 2. Print some rows

Rows come back as dicts keyed by the CSVW column names, with values already converted to Python types (datetime, float, int).

In [3]:
rows = list(table.iterdicts())
print(len(rows), "rows")
for row in rows[:5]:
    print(f"{row['timestamp']:%Y-%m-%d %H:%M}  {row['value']:>9.2f}  {row['quality_code_name']}")

rows[0]

94853 rows
2021-03-03 23:45     902.27  EstimatedA
2021-03-04 00:15     901.96  GoodExt
2021-03-04 01:30     901.08  GoodExt
2021-03-04 01:45     906.35  GoodExt
2021-03-04 04:15     906.05  GoodExt


OrderedDict([('ts_id', 289423042),
             ('timestamp',
              datetime.datetime(2021, 3, 3, 23, 45, tzinfo=datetime.timezone.utc)),
             ('value', 902.27),
             ('interpolation_type', 101),
             ('quality_code', 180),
             ('quality_code_name', 'EstimatedA'),
             ('quality_code_description',
              'Estimated (by automatic interpolation) - 180'),
             ('quality_code_color', 'rgba(255, 213, 170, 1.0)')])

## 3. To a pandas DataFrame

Because the values are already typed, pandas picks up the right dtypes without any parsing options.

In [4]:
df = pd.DataFrame(rows)
df.head()

,ts_id,timestamp,value,interpolation_type,quality_code,quality_code_name,quality_code_description,quality_code_color
0,289423042,2021-03-03 23:45:00+00:00,902.27,101,180,EstimatedA,Estimated (by automatic interpolation) - 180,"rgba(255, 213, 170, 1.0)"
1,289423042,2021-03-04 00:15:00+00:00,901.96,104,100,GoodExt,Good quality: external - 100,"rgba(0, 128, 0, 1.0)"
2,289423042,2021-03-04 01:30:00+00:00,901.08,104,100,GoodExt,Good quality: external - 100,"rgba(0, 128, 0, 1.0)"
3,289423042,2021-03-04 01:45:00+00:00,906.35,104,100,GoodExt,Good quality: external - 100,"rgba(0, 128, 0, 1.0)"
4,289423042,2021-03-04 04:15:00+00:00,906.05,104,100,GoodExt,Good quality: external - 100,"rgba(0, 128, 0, 1.0)"


In [5]:
df.dtypes

ts_id                                     int64
timestamp                   datetime64[ns, UTC]
value                                   float64
interpolation_type                        int64
quality_code                              int64
quality_code_name                        object
quality_code_description                 object
quality_code_color                       object
dtype: object

In [6]:
df["value"].describe()

count    94853.000000
mean      4429.554312
std       3731.230868
min        125.570000
25%       1409.980000
50%       3375.360000
75%       6064.570000
max      19035.510000
Name: value, dtype: float64